In [ ]:
from pathlib import Path
import json
import os
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 661
PROJECT_YEARS = list(range(2018, 2025))
PM25_THRESHOLD = 25.0
MIN_VALID_PM_HOURS = 18
MIN_VALID_COUNT_HOURS = 24
MAX_FIRE_DISTANCE_KM = 500.0
PREFILTER_DISTANCE_KM = 520.0
UPWIND_ANGLE_DEG = 45.0

# Transparent FSI research parameters. These are engineering defaults, not physical laws.
FSI_DISTANCE_TAU_KM = 100.0
FSI_AGE_HALF_LIFE_HOURS = 24.0
FSI_CONFIDENCE_WEIGHTS = {"l": 0.50, "n": 0.75, "h": 1.00}

# Optional future API acquisition. Historical Assessment 3 modelling continues to use
# the supplied 2018–2024 archive files so the evaluation period is reproducible.
USE_FIRMS_API = False
FIRMS_API_SOURCE = os.getenv("FIRMS_API_SOURCE", "VIIRS_SNPP_NRT")
FIRMS_API_BBOX = "125.5,-17.5,136.5,-7.5"  # west,south,east,north

possible_folders = [
    Path.cwd(),
    Path.cwd() / "Data Science",
    Path.home() / "Desktop" / "Data Science",
]

project_folder = None
for folder in possible_folders:
    if (folder / "ntepa_greater_darwin_hourly_2018_2024.csv").exists():
        project_folder = folder.resolve()
        break

if project_folder is None:
    raise FileNotFoundError(
        "Could not find ntepa_greater_darwin_hourly_2018_2024.csv. "
        "Place it and the seven FIRMS yearly archives (2018–2024) in the project folder."
    )

output_root = project_folder / "outputs_prt661_a3"
table_dir = output_root / "tables"
processed_dir = output_root / "processed"
for folder in [output_root, table_dir, processed_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_folder)